# 05c — per-TEMPLATE count confusion: does the `number` error have correctable structure?

> **Probe, not a rung.** Zero GPU, zero new inference. Sibling of `05b`.
> Changes no variable of the product and produces no candidate.

## Why this exists

[[the-gap-is-the-number-format]] measured that `aggregation x ID` is **80.4% `number`**, and that
no path to the target avoids lifting `number` from 0.327 to ~0.482. That leaves two live levers:
**post-hoc count calibration** (cheap; its literature support collapsed in audit) and
**synthetic-counting SFT** (two runs; the only idea with measured evidence on our backbone).

`05b` already closed the aggregate question — the model **counts but saturates at ~2**
(Spearman 0.625; acc 0.803 on the modal value vs 0.232 off it). What it could not answer, because
it pooled, is whether the error has **correctable structure**. It must be per-template:
`acc_number` is not interpretable across the 8 templates (4 degenerate, data card §3), so a LUT
fitted on the pooled slice would calibrate a mixture.

## Pre-registered reading rule — WRITTEN BEFORE LOOKING

`number` must go 0.327 -> ~0.482 (+15.5 pts) to close the gap. Calibration will not do all of it,
but it must buy a meaningful share.

| # | Condition | Verdict |
|---|---|---|
| 1 | **oracle LUT gain < +0.05** (weighted over the 2094) | calibration is **DEAD** — even the impossible-best version buys under a third of what is needed |
| 2 | oracle passes but **ID<->OOD transfer <= 0** | **DEAD** — the bias is domain-dependent and will not survive the hidden test (the risk the idea itself names) |
| 3 | **`argmax_injective == False`** on the dominant templates | mechanical proof a LUT trades one error for another (r1's stated precondition) |

Calibration survives **only** if oracle >= +0.05 **and** transfer > 0.

The oracle is fitted and evaluated on the same rows, so it **overstates** what any real calibration
could deliver. That is deliberate: it makes a small value decisive against the lever.

> ⚠️ Method guard: `local/specs/number-probe/tasks.md` records that **3 of 7 corrections in rung 05
> were interpretations bent toward the reassuring conclusion, with the data correctly produced.**
> The rule above is the countermeasure. Report numbers; read them against the table, not around it.

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parents[1]
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "experiments" / "05-bottleneck-audit"))

from _models.count_confusion import (
    confusion_by_template,
    load_number_slice,
    oracle_lut_gain,
    transfer_lut_gain,
)

# Subject: rung 06's selected checkpoint (ckpt-1720), the ladder's best scored run.
INSPECT = REPO / "experiments/06-vit-lora/runs/06_vit_lora_v1/eval_best/inspect.csv"
RUN_DIR = REPO / "experiments/05-bottleneck-audit/runs/05c_count_confusion"
RUN_DIR.mkdir(parents=True, exist_ok=True)

df = load_number_slice(INSPECT)
assert len(df) == 2094, f"expected the 2094 number questions, got {len(df)}"
assert df[["true", "pred"]].isna().sum().sum() == 0, "parser failed on some rows"
print(f"number rows: {len(df)}  |  templates: {df.template.nunique()}  |  {df.dist.value_counts().to_dict()}")

/home/legokna/miniconda3/envs/orena-frame/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


number rows: 2094  |  templates: 8  |  {'OOD': 1326, 'ID': 768}


## 1. Per-template landscape

The 8 templates, and whether a LUT is even mechanically possible on each.

In [2]:
mats, summary = confusion_by_template(df)
summary.assign(template=summary.template.str[:52]).round(4)

,template,n,n_true_levels,acc,argmax_injective,underpowered
0,How many different foreign object instances ap...,830,11,0.3361,False,False
1,How many Clips appear in this frame? Please pr...,681,12,0.2658,False,False
2,How many different foreign object classes appe...,436,4,0.6651,False,False
3,How many Sponges appear in this frame? Please ...,83,2,0.9277,False,False
4,How many External drains appear in this frame?...,45,1,0.9778,True,False
5,How many Needles appear in this frame? Please ...,9,1,1.0000,True,True
6,How many Specimens appear in this frame? Pleas...,6,1,1.0000,True,True
7,How many Specimen bags appear in this frame? P...,4,1,1.0000,True,True


## 2. Oracle LUT gain — the optimistic upper bound (rule 1)

In [3]:
oracle = oracle_lut_gain(df)
oracle_weighted = float((oracle.oracle_gain * oracle.n).sum() / oracle.n.sum())
print(f"ORACLE GAIN weighted over the 2094 = {oracle_weighted:+.4f}   (pre-registered threshold: +0.05)")
oracle.assign(template=oracle.template.str[:52]).round(4)

ORACLE GAIN weighted over the 2094 = +0.0263   (pre-registered threshold: +0.05)


,template,n,acc_before,acc_oracle,oracle_gain
0,How many different foreign object instances ap...,830,0.3361,0.3771,0.0410
1,How many Clips appear in this frame? Please pr...,681,0.2658,0.2937,0.0279
2,How many different foreign object classes appe...,436,0.6651,0.6674,0.0023
3,How many Sponges appear in this frame? Please ...,83,0.9277,0.9277,0.0000
4,How many External drains appear in this frame?...,45,0.9778,1.0000,0.0222
5,How many Needles appear in this frame? Please ...,9,1.0000,1.0000,0.0000
6,How many Specimens appear in this frame? Pleas...,6,1.0000,1.0000,0.0000
7,How many Specimen bags appear in this frame? P...,4,1.0000,1.0000,0.0000


## 3. Does it transfer across distributions? (rule 2)

In [4]:
transfer = transfer_lut_gain(df)
transfer = transfer[transfer.n_apply >= 30]
transfer_weighted = float((transfer.transfer_gain * transfer.n_apply).sum() / transfer.n_apply.sum())
print(f"TRANSFER GAIN weighted = {transfer_weighted:+.4f}   (pre-registered threshold: > 0)")
transfer.assign(template=transfer.template.str[:44]).round(4)

TRANSFER GAIN weighted = -0.0164   (pre-registered threshold: > 0)


,template,fit_on,apply_to,n_fit,n_apply,acc_before,acc_after,transfer_gain
0,How many Clips appear in this frame? Please,ID,OOD,285,396,0.2778,0.2727,-0.0051
1,How many Clips appear in this frame? Please,OOD,ID,396,285,0.2491,0.2211,-0.0281
6,How many Sponges appear in this frame? Pleas,ID,OOD,12,71,0.9437,0.9014,-0.0423
8,How many different foreign object classes ap,ID,OOD,122,314,0.6911,0.6911,0.0000
9,How many different foreign object classes ap,OOD,ID,314,122,0.5984,0.5820,-0.0164
10,How many different foreign object instances,ID,OOD,343,487,0.3860,0.3881,0.0021
11,How many different foreign object instances,OOD,ID,487,343,0.2653,0.2099,-0.0554


## 4. The mechanism — `P(pred | true)` on the dominant template

Read the modal (largest) cell of each row. Where it stops tracking the diagonal is where the
model's scale saturates; where several rows share the same modal column is where a LUT becomes
provably self-defeating.

In [5]:
key = [t for t in mats if "instances" in t][0]
(mats[key] * 100).round(0).fillna(0).astype(int)

pred,1.0,2.0,3.0,4.0,5.0,6.0,7.0,10.0
true,,,,,,,,
1.0,67,23,8,3,0,0,0,0
2.0,39,37,19,5,0,0,0,0
3.0,40,25,20,11,2,0,0,0
4.0,34,27,21,14,3,0,0,0
5.0,8,25,25,35,7,0,0,0
6.0,4,18,13,47,16,0,2,0
7.0,0,4,7,44,26,11,7,0
8.0,15,0,8,62,15,0,0,0
9.0,0,0,17,33,0,50,0,0


## 5. Verdict against the pre-registered rule

In [6]:
dominant = summary[~summary.underpowered & (summary.n_true_levels > 1)]
verdict = {
    "oracle_gain_weighted": round(oracle_weighted, 4),
    "rule1_oracle_ge_0.05": bool(oracle_weighted >= 0.05),
    "transfer_gain_weighted": round(transfer_weighted, 4),
    "rule2_transfer_gt_0": bool(transfer_weighted > 0),
    "rule3_argmax_injective_on_dominant": bool(dominant.argmax_injective.all()),
    "n": int(len(df)),
}
verdict["calibration_survives"] = verdict["rule1_oracle_ge_0.05"] and verdict["rule2_transfer_gt_0"]
pd.Series(verdict).to_frame("value")

,value
oracle_gain_weighted,0.0263
rule1_oracle_ge_0.05,False
transfer_gain_weighted,-0.0164
rule2_transfer_gt_0,False
rule3_argmax_injective_on_dominant,False
n,2094
calibration_survives,False


In [7]:
out = pd.DataFrame([verdict])
out.to_csv(REPO / "experiments/05-bottleneck-audit/RESULTS_count_confusion.csv", index=False)
summary.to_csv(RUN_DIR / "per_template_summary.csv", index=False)
oracle.to_csv(RUN_DIR / "oracle_lut_gain.csv", index=False)
transfer.to_csv(RUN_DIR / "transfer_lut_gain.csv", index=False)
mats[key].to_csv(RUN_DIR / "confusion_instances.csv")
print("written:", RUN_DIR)

written: /mnt/datos/code/ai/ORENA/proy/ORENA-Challenge-MEXICO/experiments/05-bottleneck-audit/runs/05c_count_confusion


## What this closes

**Calibration (idea 12) is a faithful negative, on both pre-registered counts.** The optimistic
oracle buys +0.026 where +0.155 is needed, and the transfer test is *negative* — a LUT fitted on
one distribution makes the other worse.

The mechanism is visible in §4: true values **2, 3 and 4 all share the same modal prediction (1)**.
A LUT can send `pred=1` to exactly one target, so correcting for one of them necessarily breaks the
other two. This is r1's stated precondition failing mechanically, not a power problem.

**What survives:** synthetic-counting SFT (idea 14) is now the only remaining lever in the group
that owns the gap. This probe does not endorse it — it removes its competitor. Its own stated risk
(rung 05 measured that `number` barely uses the image) is untouched by this result.

**Incidental:** the data card's "4 degenerate templates" is confirmed with names —
`External drains`, `Needles`, `Specimens`, `Specimen bags` each have exactly **one** true value in
val (n=45/9/6/4), so their accuracy is a constant-answer artefact, not counting skill.